In [1]:
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("pypdf").setLevel(logging.ERROR)

## RAG Pipeline : Data Ingestion → Chunking → Embedding → VectorDB → Retrieval → LLM

In [2]:
import os
import uuid
import numpy as np
from pathlib import Path
from typing import List, Dict, Any

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sentence_transformers import SentenceTransformer  # embedding model is available here
import chromadb

load_dotenv()

True

## Step 1 — Load PDFs

In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 5 PDF files to process

Processing: Language Models are Few Shot Learners.pdf
  ✓ Loaded 75 pages

Processing: BERT Pretraining.pdf
  ✓ Loaded 16 pages

Processing: RAG for Knowledge Intensive NLP Task.pdf
  ✓ Loaded 19 pages

Processing: Efficient Estimation of Word Rep in Vector Space.pdf
  ✓ Loaded 12 pages

Processing: Attention is all you Need.pdf
  ✓ Loaded 15 pages

Total documents loaded: 137


## Step 2 — Chunking

In [4]:
### Text splitting get into chunks using RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:100]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

chunks = split_documents(all_pdf_documents)

Split 137 documents into 597 chunks

Example chunk:
Content: Language Models are Few-Shot Learners
Tom B. Brown∗ Benjamin Mann∗ Nick Ryder∗ Melanie Subbiah∗
Jare...
Metadata: {'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2020-07-24T00:04:08+00:00', 'author': '', 'keywords': '', 'moddate': '2020-07-24T00:04:08+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/Language Models are Few Shot Learners.pdf', 'total_pages': 75, 'page': 0, 'page_label': '1', 'source_file': 'Language Models are Few Shot Learners.pdf', 'file_type': 'pdf'}


## Step 3 — Embedding Manager

In [5]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args: texts: List of text strings to embed
        Returns: numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initialize the embedding manager
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 17236.41it/s]


Model loaded successfully. Embedding dimension: 384


## Step 4 — Vector Store
A vector store is a database that stores embeddings and retrieves similar text based on meaning, not exact words

Vector stores are used to:
* store document chunks
* retrieve relevant chunks for the LLM

In [6]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)  # Save embeddings on disk
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG",
                    "hnsw:space": "cosine"  # Use cosine distance so similarity = 1 - distance
                }
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        For each document:
            - Generates unique ID
            - Stores text (page_content)
            - Stores metadata (source info, length, etc.)
            - Stores embedding vector

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [7]:
# # To Delete the old persisted vector store, if new documents are uploaded
# import shutil
# shutil.rmtree("../data/vector_store", ignore_errors=True)
# print("Old vector store deleted")

## Step 4b — Embed Chunks and Index into Vector Store

In [8]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 597 texts...


Batches: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 19/19 [00:02<00:00,  8.08it/s]


Generated embeddings with shape: (597, 384)
Adding 597 documents to vector store...
Successfully added 597 documents to vector store
Total documents in collection: 597


## Step 5 — Retriever Pipeline From VectorStore

User Query → Convert query to embedding → Search vector database → Return most similar document chunks

In [9]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
            vector_store: Vector store containing document embeddings, eg Chunk 1 -> [0.1, 0.4, ...] stored in ChromaDB
            embedding_manager: Used to convert user queries into embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager  # Used to convert user queries into embeddings. Ex : "What is a transformer model?" becomes [0.42, -0.15, 0.91, ...]


    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store --- This performs vector similarity search.
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

## Step 6 — LLM : Query Retrieval Pipeline - VectorDB + LLM Output Generation

In [10]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str = None):
        """
        Initialize Groq LLM

        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError("Groq API key is required.")

        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized Groq LLM with model: {self.model_name}")

    # "context" is the relevant text chunks you pulled from your documents and are handing to the LLM alongside the question.
    # context generated using :: `rag_retriever.retrieve("..")`
    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        Returns:
            Generated response string
        """

        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context as your primary source, and supplement with your own knowledge where the context is incomplete.
            Context:
            {context}

            Question: {question}

            Answer:"""
        )

        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)

        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content

        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}
        Question: {query}
        Answer:"""

        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

llm = GroqLLM(model_name="llama-3.3-70b-versatile")

Initialized Groq LLM with model: llama-3.3-70b-versatile


## Step 7 — Advanced RAG Pipeline: Citations, History, Summarization

In [11]:
from typing import Dict, Any

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        summarize: bool = False
    ) -> Dict[str, Any]:

        results = self.retriever.retrieve(
            question,
            top_k=top_k,
            score_threshold=min_score
        )

        if not results:
            answer = "No relevant context found."
            sources = []
        else:
            context = "\n\n".join([doc["content"] for doc in results])

            sources = [
                {
                    "source": doc["metadata"].get(
                        "source_file",
                        doc["metadata"].get("source", "unknown")
                    ),
                    "page": doc["metadata"].get("page", "unknown"),
                    "score": round(doc["similarity_score"], 3),
                    "preview": doc["content"][:120] + "..."
                }
                for doc in results
            ]

            answer = self.llm.generate_response(
                query=question,
                context=context
            )

        # Add citations
        citations = [
            f"[{i+1}] {src['source']} (page {src['page']})"
            for i, src in enumerate(sources)
        ]
        answer_with_citations = (
            answer + "\n\nCitations:\n" + "\n".join(citations)
            if citations
            else answer
        )

        # Optional summarization — no document context needed here
        summary = None
        if summarize and answer:
            summary = self.llm.generate_response_simple(
                query=f"Summarize the following answer in 2 concise sentences: {answer}",
                context=""
            )

        self.history.append({
            "question": question,
            "answer": answer,
            "sources": sources,
            "summary": summary
        })

        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history
        }

adv_rag = AdvancedRAGPipeline(retriever=rag_retriever, llm=llm)

## Run Queries

In [17]:
query = "If you were building a question answering system today, which components from the Transformer, BERT, and RAG papers would you combine and why?"

In [18]:
result = adv_rag.query(
    question=query,
    top_k=3,
    min_score=0.1,
    summarize=True
)

Retrieving documents for query: 'If you were building a question answering system today, which components from the Transformer, BERT, and RAG papers would you combine and why?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 16.46it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


In [21]:
def print_rag_result(result):
    width = 80

    print("=" * width)
    print("QUESTION")
    print("=" * width)
    print(result["question"])

    print("\n" + "=" * width)
    print("ANSWER")
    print("=" * width)
    print(result["answer"])

    if result["summary"]:
        print("\n" + "=" * width)
        print("SUMMARY")
        print("=" * width)
        print(result["summary"])

    print("\n" + "=" * width)
    print("SOURCES")
    print("=" * width)
    for source in result["sources"]:
        print(f"  [{source['source']}]  Page {source['page']}  |  Score: {source['score']}")
        print(f"  Preview: {source['preview']}")
        print("-" * width)

print_rag_result(result)

QUESTION
If you were building a question answering system today, which components from the Transformer, BERT, and RAG papers would you combine and why?

ANSWER
If I were building a question answering system today, I would combine components from the Transformer, BERT, and RAG papers as follows:

1. **Transformer Architecture**: I would use the Transformer architecture as the foundation for my question answering system. The Transformer's self-attention mechanism and encoder-decoder structure have proven to be highly effective in natural language processing tasks, including question answering.
2. **BERT-based Encoder**: I would use a BERT-based encoder, such as BERT-base or BART, to encode the input question and documents. BERT has demonstrated state-of-the-art performance in many NLP tasks, and its pre-trained representations can provide a strong foundation for my question answering system.
3. **RAG's Generative Approach**: I would adopt RAG's generative approach to question answering, 